In [ ]:
# imports
import sys
import os

# Support repo root, parent of repo, or cwd=notebooks/ (jupyter / nbconvert from Submit_metrics.sh)
_REPO_NAME = 'Physically-conditioned-latent-diffusion-model-for-temperature'
_cwd = os.getcwd()
if os.environ.get('PROJECT_ROOT', '').strip():
    os.chdir(os.environ['PROJECT_ROOT'].strip())
elif os.path.basename(_cwd) == _REPO_NAME:
    pass
elif os.path.isdir(os.path.join(_cwd, _REPO_NAME)):
    os.chdir(os.path.join(_cwd, _REPO_NAME))
elif os.path.basename(_cwd) == 'notebooks' and os.path.basename(os.path.dirname(_cwd)) == _REPO_NAME:
    os.chdir(os.path.dirname(_cwd))
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm import tqdm
import pickle

import lightning as L
seed = 42
L.seed_everything(seed, workers=True)

import xarray as xr
import xskillscore as xs
import warnings

from src.models.unet_module import UnetLitModule
from src.models.gan_module import UnetGANLitModule
from src.models.ae_module import AutoencoderKL, EncoderLRES
from src.models.ldm_module import LatentDiffusion

from src.models.components.unet import DownscalingUnet
from src.models.components.ae import SimpleConvDecoder, SimpleConvEncoder
from src.models.components.ldm.denoiser import UNetModel, DDIMSampler
from src.models.components.ldm.conditioner import AFNOConditionerNetCascade
from src.data.downscaling_datamodule import DownscalingDataModule
from src.data.components.downscaling_dataset import DownscalingDataset

from utils.inference_utils import get_model_output
from utils.plotting_utils import get_target_grid, show_metrics, from_torchtensor_to_xarray

print('Repo root:', os.getcwd())

In [ ]:
# Setting paths to data (relative to repo root; see imports cell)

# results_file_path = './outputs/results_pde.pkl'
results_file_path = './outputs/Our_results_trained_models_2mT.pkl'
pretrained_metrics_file_path = './pretrained_models/outputs/metrics_trained_models.pkl'
output_path = './outputs/'

for label, path in [
    ('results', results_file_path),
    ('pretrained metrics', pretrained_metrics_file_path),
]:
    if not os.path.isfile(path):
        raise FileNotFoundError(f'Missing {label} file: {path} (cwd={os.getcwd()})')
print('Using results:', results_file_path)

In [ ]:
#Loading in dataframes

results_df = pd.read_pickle(results_file_path)
pretrained_metrics_df = pd.read_pickle(pretrained_metrics_file_path)

In [ ]:
#printing head and tails

print(results_df.columns)
print(results_df.shape)
print(results_df.head())
print(results_df.tail())

print_metrics = True
if print_metrics:
    print(f'\n\n =======================Pretrained Metrics below======================= \n\n')

    print(pretrained_metrics_df.columns)
    print(pretrained_metrics_df.shape)
    print(pretrained_metrics_df.head())
    print(pretrained_metrics_df.tail())
    print(pretrained_metrics_df['metric'].unique().tolist())

In [ ]:
# split results_df into separate dataframes, one per model

# Use None for all models in the file, or choose a subset/order (same pattern as plot_predictions_by_timeslice.ipynb)
# model_order = ['ERA5', 'COSMO-CLM', 'UNET', 'GAN', 'LDM_res', 'LDM_PDE_res', 'LMM_PDE_res']
model_order = ['Quadratic Interp.', 'UNET', 'GAN', 'LDM_res', 'LDM_PDE_res', 'LMM_PDE_res_020_1_inv_ni', 'LMM_PDE_res_033_1_inv_ni']

# Exclude models from metrics/plots, for example: ['ERA5']
exclude_models = []

# Reference fields always loaded (not in model_order): COSMO-CLM = HR target, ERA5 = coarse input for FluxRatio
REFERENCE_MODELS = ['COSMO-CLM', 'ERA5']


def resolve_models(df, model_order=None, exclude=None):
    available = list(df['model'].drop_duplicates())
    models = available if model_order is None else list(model_order)
    exclude = set(exclude or [])
    models = [model for model in models if model not in exclude]

    missing_models = [model for model in models if model not in set(available)]
    if missing_models:
        raise ValueError(f'Missing requested models: {missing_models}')
    if not models:
        raise ValueError('No models left to plot after applying exclude_models.')
    return models


available_models = list(results_df['model'].drop_duplicates())
print(f'Available models ({len(available_models)}):')
print(available_models)

model_types = resolve_models(results_df, model_order=model_order, exclude=exclude_models)
print(f'Models for metrics/plots: {model_types}')

models_for_results = list(dict.fromkeys(model_types + [m for m in REFERENCE_MODELS if m in available_models]))

results = {}
for model_type in models_for_results:
    model_df = results_df[results_df['model'] == model_type]
    model_df = model_df.reset_index(drop=True)
    results[model_type] = model_df

print(results.keys())

In [ ]:
# Computing metrics for each model
metric_models = [model for model in model_types if model not in ['ERA5', 'COSMO-CLM']]
metrics_list = []

target_grid_high_res = get_target_grid('high')

for model_i in metric_models:
    cosmo_df = results['COSMO-CLM']
    for _, model_i_row in results[model_i].iterrows():
        cosm_sel = cosmo_df[
            (cosmo_df['time_step'] == model_i_row['time_step'])
            & (cosmo_df['input_var'] == model_i_row['input_var'])
            & (cosmo_df['target_var'] == model_i_row['target_var'])
            & (cosmo_df['variable'] == model_i_row['variable'])
        ]
        if len(cosm_sel) != 1:
            raise ValueError(
                f"Expected one COSMO-CLM row for time_step={model_i_row['time_step']!r} "
                f"model={model_i!r} var={model_i_row['variable']!r}; got {len(cosm_sel)}"
            )
        cosmo_row = cosm_sel.iloc[0]

        cosmo_spat_distr = cosmo_row['spat_distr']
        model_i_spat_distr = model_i_row['spat_distr']

        cosmo_xr = from_torchtensor_to_xarray(cosmo_spat_distr, target_grid_high_res, coords_name='y_x')
        model_i_xr = from_torchtensor_to_xarray(model_i_spat_distr, target_grid_high_res, coords_name='y_x')

        # Computing the 5 metrics listed in Appendix B (+ CRPS)
        model_i_rmse = xs.rmse(model_i_xr, cosmo_xr).item() # RMSE
        model_i_me = xs.me(model_i_xr, cosmo_xr).item() # BIAS (mean error)
        model_i_r2 = xs.r2(model_i_xr, cosmo_xr).item() # coefficient of determination
        model_i_pearson = xs.pearson_r(model_i_xr, cosmo_xr).item() # Pearson correlation
        # Deterministic forecasts: 1-member ensemble CRPS equals MAE.
        model_i_crps = xs.crps_ensemble(
            cosmo_xr, model_i_xr.expand_dims(member=[0])
        ).item()

        # Appending the metrics to the list
        model_i_metrics = {
            'RMSE': model_i_rmse,
            'R2': model_i_r2,
            'BIAS': model_i_me,
            'PCC': model_i_pearson,
            'CRPS': model_i_crps,
        }

        for model_i_metric in model_i_metrics:
            metrics_list.append({
                'model': model_i,
                'target_var': model_i_row['target_var'],
                'var': model_i_row['variable'],
                'metric': model_i_metric,
                'value': model_i_metrics[model_i_metric],
            })

metrics_df = pd.DataFrame(metrics_list)
metrics_df = metrics_df.reset_index(drop=True)

print(metrics_df)
metrics_df.to_pickle(output_path + './Our_inference_metrics' + '.pkl')  

In [ ]:
# Compute and Plot Super-Grid PDE Flux Ratio Metric 

import torch
import numpy as np


# GPU-accelerated supercell flux ratio code 

def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def compute_gradients_torch(T, dx=1.0, dy=1.0, eps=1e-6):
    H, W = T.shape
    dTdx = torch.zeros_like(T)
    dTdy = torch.zeros_like(T)
    # Interior
    if W > 2:
        dTdx[:, 1:-1] = (T[:, 2:] - T[:, :-2]) / (2.0 * dx)
    if H > 2:
        dTdy[1:-1, :] = (T[2:, :] - T[:-2, :]) / (2.0 * dy)
    # Boundaries
    if W > 1:
        dTdx[:, 0] = (T[:, 1] - T[:, 0]) / dx
        dTdx[:, -1] = (T[:, -1] - T[:, -2]) / dx
    if H > 1:
        dTdy[0, :] = (T[1, :] - T[0, :]) / dy
        dTdy[-1, :] = (T[-1, :] - T[-2, :]) / dy
    return dTdx, dTdy

def compute_block_effective_flux_ratio_torch(
    T_block, dTdx_block, dTdy_block, dx=1.0, dy=1.0, eps=1e-6
):
    device = T_block.device
    H, W = T_block.shape
    adv_vals, diff_vals = [], []
    
    def process_edge(i_idx, j_idx, nx, ny):
        grad_x = dTdx_block[i_idx, j_idx]
        grad_y = dTdy_block[i_idx, j_idx]
        Tvals  = T_block[i_idx, j_idx]
        grad_norm = torch.sqrt(grad_x**2 + grad_y**2) + eps
        g_hat_x = grad_x / grad_norm
        g_hat_y = grad_y / grad_norm
        dot = g_hat_x * nx + g_hat_y * ny
        adv_ = Tvals * dot
        diff_ = torch.sqrt(grad_x**2 + grad_y**2)
        return adv_, diff_
    
    # Top Edge
    if H > 0:
        i_top = torch.zeros(W, dtype=torch.long, device=device)
        j_top = torch.arange(W, device=device)
        adv, dif = process_edge(i_top, j_top, nx=0.0, ny=-1.0)
        adv_vals.append(adv)
        diff_vals.append(dif)
    # Bottom Edge
    if H > 1:
        i_bot = torch.full((W,), H - 1, dtype=torch.long, device=device)
        j_bot = torch.arange(W, device=device)
        adv, dif = process_edge(i_bot, j_bot, nx=0.0, ny=1.0)
        adv_vals.append(adv)
        diff_vals.append(dif)
    # Left Edge
    if W > 0 and H > 2:
        i_left = torch.arange(1, H - 1, device=device)
        j_left = torch.zeros(H - 2, dtype=torch.long, device=device)
        adv, dif = process_edge(i_left, j_left, nx=-1.0, ny=0.0)
        adv_vals.append(adv)
        diff_vals.append(dif)
    # Right Edge
    if W > 1 and H > 2:
        i_right = torch.arange(1, H - 1, device=device)
        j_right = torch.full((H - 2,), W - 1, dtype=torch.long, device=device)
        adv, dif = process_edge(i_right, j_right, nx=1.0, ny=0.0)
        adv_vals.append(adv)
        diff_vals.append(dif)
    
    if len(adv_vals) == 0:
        return 0.0

    adv_all = torch.cat(adv_vals)
    diff_all = torch.cat(diff_vals)
    return (torch.mean(adv_all) / (torch.mean(diff_all) + eps)).item()

def compute_supercell_flux_ratio_field(
    T, supercell_size=16, dx=1.0, dy=1.0, eps=1e-6
):
    device = get_device()
    if not isinstance(T, torch.Tensor):
        T_torch = torch.from_numpy(T).float().to(device)
    else:
        T_torch = T.float().to(device)
    H, W = T_torch.shape
    dTdx, dTdy = compute_gradients_torch(T_torch, dx=dx, dy=dy, eps=eps)
    num_cells_vert = H // supercell_size
    num_cells_horiz = W // supercell_size
    R_eff_field = torch.zeros((num_cells_vert, num_cells_horiz), device=device)
    for i in range(num_cells_vert):
        for j in range(num_cells_horiz):
            r0, r1 = i*supercell_size, (i+1)*supercell_size
            c0, c1 = j*supercell_size, (j+1)*supercell_size
            T_block    = T_torch[r0:r1, c0:c1]
            dTdx_block = dTdx[r0:r1, c0:c1]
            dTdy_block = dTdy[r0:r1, c0:c1]
            R_eff = compute_block_effective_flux_ratio_torch(T_block, dTdx_block, dTdy_block, dx=dx, dy=dy, eps=eps)
            R_eff_field[i, j] = R_eff
    return R_eff_field.cpu().numpy()

def compute_flux_ratio_loss_supercell(ref_field, pred_field, supercell_size=16, dx=1.0, dy=1.0, eps=1e-6):
    R_ref = compute_supercell_flux_ratio_field(ref_field, supercell_size, dx, dy, eps)
    R_pred = compute_supercell_flux_ratio_field(pred_field, supercell_size, dx, dy, eps)
    # If shapes differ, take overlap
    min_h = min(R_ref.shape[0], R_pred.shape[0])
    min_w = min(R_ref.shape[1], R_pred.shape[1])
    R_ref = R_ref[:min_h, :min_w]
    R_pred = R_pred[:min_h, :min_w]
    # PDE-style flux ratio difference = mean absolute difference
    return np.mean(np.abs(R_pred - R_ref))


# Compute super-grid flux ratio metric for each model/time/2mT
fluxratio_metrics_list = []
for model_i in metric_models:
    # Only do flux ratio for 2mT
    model_rows_2mt = results[model_i][results[model_i]['variable'] == '2mT']
    for idx, row in model_rows_2mt.iterrows():
        ts = row['time_step']
        # Matching ERA5 row => reference coarse
        era5_row = results['ERA5'][(results['ERA5']['time_step'] == ts) &
                                   (results['ERA5']['variable'] == '2mT')]
        if era5_row.empty:
            continue
        # Numpy or array data
        T_c = era5_row.iloc[0]['spat_distr'].numpy() if hasattr(era5_row.iloc[0]['spat_distr'],'numpy') \
             else era5_row.iloc[0]['spat_distr']
        T_f = row['spat_distr'].numpy() if hasattr(row['spat_distr'],'numpy') else row['spat_distr']
        
        flux_loss = compute_flux_ratio_loss_supercell(T_c, T_f, supercell_size=16, dx=1.0, dy=1.0, eps=1e-6)
        fluxratio_metrics_list.append({
            'model': model_i,
            'target_var': row['target_var'],
            'var': row['variable'],
            'metric': 'FluxRatio',
            'value': flux_loss,
        })

if fluxratio_metrics_list:
    fluxratio_df = pd.DataFrame(fluxratio_metrics_list)
    metrics_df = pd.concat([metrics_df, fluxratio_df], ignore_index=True)

print("[Done] Super-grid PDE flux ratio metrics computed and appended to metrics_df.")
metrics_df.to_pickle(output_path + './Our_inference_metrics_with_PDE.pkl')


# Plot the distribution of "FluxRatio" metric, similar to other metrics

import seaborn as sns
sns.set_theme(font_scale=1.2, style="whitegrid")

fluxratio_plot_df = metrics_df[metrics_df['metric'] == 'FluxRatio'].copy()
if len(fluxratio_plot_df) == 0:
    print("No 'FluxRatio' metrics found in metrics_df. Skipping super-grid PDE flux ratio plot.")
else:
    g = sns.catplot(
        data=fluxratio_plot_df, kind='box',
        x='model', y='value', hue='model',
        height=5, aspect=1.5, legend=False,
        showmeans=True,
        meanprops={'marker': 'v', 'markerfacecolor': 'white', 'markeredgecolor': 'black', 'markersize': 7}
    )
    for ax in g.axes.flatten():
        for label in ax.get_xticklabels():
            label.set_rotation(45)
        ax.set_xlabel("")
        ax.set_title("Super-grid PDE Flux Ratio Loss (2mT)")
        ax.set_ylabel("Flux Ratio Loss")
    
    # Adjust legend, remove duplicates
    if g._legend is not None:
        g._legend.remove()
    plt.tight_layout()
    plt.show()
# Append the flux ratio metrics to the existing metrics DataFrame
if fluxratio_metrics_list:
    fluxratio_df = pd.DataFrame(fluxratio_metrics_list)
    metrics_df = pd.concat([metrics_df, fluxratio_df], ignore_index=True)

# Save the updated metrics DataFrame to file
metrics_df.to_pickle(output_path + './Our_inference_metrics_with_PDE.pkl')

# (Optional) Print the head of the updated dataframe to verify the new entries
print(metrics_df.head(10))



In [ ]:
# Plotting metrics (combined figure: RMSE, R2, BIAS, PCC, CRPS)
COMBINED_METRICS = ['RMSE', 'R2', 'BIAS', 'PCC', 'CRPS']
METRIC_Y_REF = {'RMSE': 0, 'R2': 1, 'BIAS': 0, 'PCC': 1, 'CRPS': 0}


def show_mT_metrics(metrics, output_dir, save_to_file, metrics_to_plot=None):
    """
    Slightly modified version of show_metrics from utils.plotting_utils.
    By default plots the combined metric panel (CRPS replaces FluxRatio).
    """
    metrics_to_plot = metrics_to_plot or COMBINED_METRICS
    plot_df = metrics[metrics['metric'].isin(metrics_to_plot)].copy()
    if plot_df.empty:
        raise ValueError(
            f'No rows for combined metrics {metrics_to_plot}. '
            f'Available: {sorted(metrics["metric"].unique())}'
        )
    plot_df['metric'] = pd.Categorical(
        plot_df['metric'], categories=metrics_to_plot, ordered=True
    )

    # Set up plotting resources
    box_palette = {
        'Quadratic Interp.': 'r',
        'Linear Interp.': '#17becf',
        'UNET': 'b',
        'GAN': 'g',
        'VAE_res': 'pink',
        'LDM_res': 'orange',
        'LDM_PDE_res': 'blue',
        'LMM_PDE_res': '#9467bd'
    }
    # Keep all LMM variants (e.g., LMM_PDE_res_steps_1/3/5) in the same color.
    lmm_color = box_palette['LMM_PDE_res']
    for model_name in plot_df['model'].unique():
        if str(model_name).startswith('LMM_PDE_res'):
            box_palette[model_name] = lmm_color

    # Plot boxplots
    sns.set_theme(font_scale=1.5, style="whitegrid")
    g = sns.catplot(
        data=plot_df,
        kind='box',
        x='model',
        y='value',
        col='metric',
        col_order=metrics_to_plot,
        row='var',
        hue='model',
        native_scale=True,
        sharey=False,
        margin_titles=True,
        palette=box_palette,
        showmeans=True,
        meanprops={
            'marker': 'v',
            'markerfacecolor': 'w',
            'markeredgecolor': 'black',
            'markersize': '8',
        },
    )
    n_metrics = len(metrics_to_plot)
    for i, ax in enumerate(g.axes.flat):
        for label in ax.get_xticklabels():
            label.set_rotation(90)
        metric_name = metrics_to_plot[i % n_metrics]
        ax.axline(
            (0, METRIC_Y_REF[metric_name]),
            slope=0,
            linestyle='--',
            color='gray',
            linewidth=3,
        )
        ax.set(xlabel=None)
    if g._legend is not None:
        g._legend.remove()

    if save_to_file:
        g.savefig(output_dir + 'Fig_metrics.jpg')

show_mT_metrics(metrics_df, output_path, False)

In [ ]:
# --- Cell 9 (code) ---
import numpy as np


# Helper functions

def _radial_spectral_slope(field: np.ndarray):
    """Return log-log slope of radially averaged 2-D power spectrum."""
    field = np.squeeze(field) - field.mean()
    H, W   = field.shape
    P      = np.abs(np.fft.fftshift(np.fft.fft2(field)))**2

    # radial distances (pixel units) from spectrum centre
    y, x   = np.indices((H, W))
    r      = np.sqrt((y - H/2)**2 + (x - W/2)**2).astype(int)

    max_r  = min(H, W)//2
    radial_mean = np.bincount(r.ravel(), P.ravel()) / np.maximum(1, np.bincount(r.ravel()))
    radial_mean = radial_mean[1:max_r]      # skip DC component (r=0)
    radii       = np.arange(1, len(radial_mean)+1)

    # keep only bins with positive power
    mask = radial_mean > 0
    if mask.sum() < 3:              # spectrum too short for a fit
        return np.nan
    log_k = np.log10(radii[mask])
    log_P = np.log10(radial_mean[mask])
    slope, _ = np.polyfit(log_k, log_P, 1)
    return slope                    

def _spectral_slope_error(ref, pred):
    s_ref  = _radial_spectral_slope(ref)
    s_pred = _radial_spectral_slope(pred)
    return np.abs(s_pred - s_ref)


# Loop through models/timesteps and compute the spectral slope difference

physical_metrics = []
for model_name in metric_models:                      # defined earlier
    rows_2mt = results[model_name][results[model_name]['variable'] == '2mT']
    for _, row in rows_2mt.iterrows():
        ts = row['time_step']
        ref_row = results['COSMO-CLM'][
            (results['COSMO-CLM']['time_step'] == ts) &
            (results['COSMO-CLM']['variable']   == '2mT')
        ]
        if ref_row.empty:
            continue

        ref_field  = ref_row.iloc[0]['spat_distr']
        pred_field = row['spat_distr']
        ref_np     = ref_field.numpy()  if hasattr(ref_field,  'numpy') else ref_field
        pred_np    = pred_field.numpy() if hasattr(pred_field, 'numpy') else pred_field

        # spectral slope difference
        slope_err = _spectral_slope_error(ref_np, pred_np)

        physical_metrics.append({
            'model':      model_name,
            'target_var': row['target_var'],
            'var':        row['variable'],
            'metric':     'SpectralSlopeDiff',
            'value':      slope_err
        })

# Append and save
physical_df = pd.DataFrame(physical_metrics)
metrics_df  = pd.concat([metrics_df, physical_df], ignore_index=True)
metrics_df.to_pickle(output_path + './Our_inference_metrics_with_PDE_Phys.pkl')

print("✔ Added SpectralSlopeDiff (lower = better).")


In [ ]:

import seaborn as sns
import matplotlib.pyplot as plt

# only keep the slope‐difference metric
plot_metrics = ['SpectralSlopeDiff']
plot_df = metrics_df[metrics_df['metric'].isin(plot_metrics)].copy()

sns.set_theme(style="whitegrid", font_scale=1.3)
g = sns.catplot(
    data      = plot_df,
    kind      = 'box',
    x         = 'model',
    y         = 'value',
    hue       = 'model',
    col       = 'metric',
    row       = 'var',
    sharey    = False,
    height    = 5,
    aspect    = 1.3,
    showmeans = True,
    meanprops = {'marker':'v','markerfacecolor':'white',
                 'markeredgecolor':'black','markersize':8}
)

for ax in g.axes.flatten():
    ax.axhline(0, ls='--', lw=2, color='gray')  # ideal line at zero
    ax.set_xlabel('')
    for tick in ax.get_xticklabels():
        tick.set_rotation(45)

if g._legend is not None:
    g._legend.remove()

plt.tight_layout()
plt.show()


In [ ]:
# Anisotropic transport metrics (TemperatureFieldLosses + downscaling_LMM_res_2mT_pretrain.yaml)
#
# Mirrors src/models/lmm_module.py and temperature_field_losses.py:
#   L_mag, L_dir_cosine, L_dir_unit_mse, L_dir, L_at (= L_total)
#
# Merged figure: L_mag + both L_dir components + combined L_dir (λ=1 for visualization).
# L_at figure: yaml-weighted L_total (λ_mag=0.01·L_mag + L_dir). Mask: at_qmag_min=1e-15.

import torch
import seaborn as sns
import matplotlib.pyplot as plt
from src.models.temperature_field_losses import TemperatureFieldLosses
from utils.temperature_normalization import (
    COSMO_SOURCE,
    default_normalization_pickle,
    load_norm_stats,
    normalize_temperature,
)

field_losses = TemperatureFieldLosses()

# --- configs/experiment/downscaling_LMM_res_2mT_pretrain.yaml ---
AT_LAMBDA_MAG = 0.01
AT_LAMBDA_DIR_COSINE = 1.0
AT_LAMBDA_DIR_UNIT_MSE = 1.0
AT_LOSS_EPS = 1.0e-15
AT_DX, AT_DY = 2000.0, -2000.0
AT_QMAG_QUANTILE = None
AT_QMAG_MIN = 1.0e-15

# Merged panel: unit weights (visualization only)
AT_VIZ_LAMBDA_MAG = 1.0
AT_VIZ_LAMBDA_DIR_COSINE = 1.0
AT_VIZ_LAMBDA_DIR_UNIT_MSE = 1.0

AT_MERGED_METRICS = ['L_mag', 'L_dir_cosine', 'L_dir_unit_mse', 'L_dir']
AT_MERGED_LABELS = {
    'L_mag': 'L_mag',
    'L_dir_cosine': 'L_dir (cosine)',
    'L_dir_unit_mse': 'L_dir (unit MSE)',
    'L_dir': 'L_dir = L_dir_cosine + L_dir_unit_mse (λ=1)',
    'L_at': f'L_at = {AT_LAMBDA_MAG}·L_mag + L_dir (yaml)',
}

_norm_values = load_norm_stats(default_normalization_pickle())


def _temperature_tensor(field) -> torch.Tensor:
    arr = field.numpy() if hasattr(field, 'numpy') else field
    arr_norm = normalize_temperature(
        arr, variable='2mT', source=COSMO_SOURCE, norm_values=_norm_values
    )
    return torch.as_tensor(arr_norm, dtype=torch.float32).unsqueeze(0)


def _at_loss_kwargs():
    return dict(
        dx=AT_DX,
        dy=AT_DY,
        eps=AT_LOSS_EPS,
        qmag_quantile=AT_QMAG_QUANTILE,
        qmag_min=AT_QMAG_MIN,
    )


def plot_at_metric_boxplots(
    df,
    metric_names,
    *,
    col_wrap=None,
    aspect=1.25,
    ylabel='loss (lower = better)',
):
    plot_df = df[df['metric'].isin(metric_names)].copy()
    if plot_df.empty:
        print(f'No metrics found for {metric_names}. Skipping plot.')
        return

    plot_df['metric'] = pd.Categorical(
        plot_df['metric'], categories=metric_names, ordered=True
    )

    sns.set_theme(style='whitegrid', font_scale=1.3)
    g = sns.catplot(
        data=plot_df,
        kind='box',
        x='model',
        y='value',
        hue='model',
        col='metric',
        col_order=metric_names,
        col_wrap=col_wrap,
        sharey=False,
        height=5,
        aspect=aspect,
        legend=False,
        showmeans=True,
        meanprops={
            'marker': 'v',
            'markerfacecolor': 'white',
            'markeredgecolor': 'black',
            'markersize': 7,
        },
    )

    for ax, metric_name in zip(g.axes.flatten(), metric_names):
        ax.axhline(0, ls='--', lw=2, color='gray')
        ax.set_xlabel('')
        ax.set_ylabel(ylabel)
        ax.set_title(AT_MERGED_LABELS.get(metric_name, metric_name))
        for tick in ax.get_xticklabels():
            tick.set_rotation(45)

    plt.tight_layout()
    plt.show()


at_metrics_list = []
at_mask_fracs = []
for model_name in metric_models:
    rows_2mt = results[model_name][results[model_name]['variable'] == '2mT']
    for _, row in rows_2mt.iterrows():
        ref_row = results['COSMO-CLM'][
            (results['COSMO-CLM']['time_step'] == row['time_step'])
            & (results['COSMO-CLM']['variable'] == '2mT')
        ]
        if ref_row.empty:
            continue

        T_gt = _temperature_tensor(ref_row.iloc[0]['spat_distr'])
        T_pred = _temperature_tensor(row['spat_distr'])

        with torch.no_grad():
            at_viz = field_losses.anisotropic_transport_loss(
                T_pred,
                T_gt,
                lambda_mag=AT_VIZ_LAMBDA_MAG,
                lambda_dir_cosine=AT_VIZ_LAMBDA_DIR_COSINE,
                lambda_dir_unit_mse=AT_VIZ_LAMBDA_DIR_UNIT_MSE,
                **_at_loss_kwargs(),
            )
            at_train = field_losses.anisotropic_transport_loss(
                T_pred,
                T_gt,
                lambda_mag=AT_LAMBDA_MAG,
                lambda_dir_cosine=AT_LAMBDA_DIR_COSINE,
                lambda_dir_unit_mse=AT_LAMBDA_DIR_UNIT_MSE,
                **_at_loss_kwargs(),
            )

        row_base = {
            'model': model_name,
            'target_var': row['target_var'],
            'var': row['variable'],
        }
        for metric_name, tensor_key in [
            ('L_mag', 'L_mag'),
            ('L_dir_cosine', 'L_dir_cosine'),
            ('L_dir_unit_mse', 'L_dir_unit_mse'),
            ('L_dir', 'L_dir'),
        ]:
            at_metrics_list.append(
                {
                    **row_base,
                    'metric': metric_name,
                    'value': float(at_viz[tensor_key].item()),
                }
            )
        at_metrics_list.append(
            {
                **row_base,
                'metric': 'L_at',
                'value': float(at_train['L_total'].item()),
            }
        )
        if 'at_mask_frac' in at_viz:
            at_mask_fracs.append(float(at_viz['at_mask_frac'].item()))

if at_metrics_list:
    at_df = pd.DataFrame(at_metrics_list)
    metrics_df = pd.concat([metrics_df, at_df], ignore_index=True)
    metrics_df.to_pickle(output_path + './Our_inference_metrics_with_at.pkl')

mean_mask_frac = np.mean(at_mask_fracs) if at_mask_fracs else float('nan')
print(
    f'✔ Added anisotropic transport metrics ({len(at_metrics_list)} rows). '
    f'Mean GT |q| mask fraction kept: {mean_mask_frac:.4f} '
    f'(qmag_quantile={AT_QMAG_QUANTILE}, qmag_min={AT_QMAG_MIN}).'
)
print(
    f'   Merged panel λ: mag={AT_VIZ_LAMBDA_MAG}, dir_cosine={AT_VIZ_LAMBDA_DIR_COSINE}, '
    f'dir_unit_mse={AT_VIZ_LAMBDA_DIR_UNIT_MSE}'
)
print(
    f'   L_at yaml λ: mag={AT_LAMBDA_MAG}, dir_cosine={AT_LAMBDA_DIR_COSINE}, '
    f'dir_unit_mse={AT_LAMBDA_DIR_UNIT_MSE}'
)

plot_at_metric_boxplots(
    metrics_df,
    AT_MERGED_METRICS,
    aspect=1.15,
    ylabel='anisotropic loss (lower = better)',
)

plot_at_metric_boxplots(
    metrics_df,
    ['L_at'],
    aspect=1.5,
    ylabel=AT_MERGED_LABELS['L_at'],
)


In [ ]:
# Flux VECTOR absolute error (MAE style) with GT |q| mask
#
# Analogous to FluxRatio's mean absolute difference, but for vector flux q=(qx,qy):
#   Q_vec_L1_MAE_masked = mean_mask( (|qx_pred - qx_gt| + |qy_pred - qy_gt|) / 2 )
#
# Mask follows training anisotropic transport logic in TemperatureFieldLosses:
# - qmag_quantile: keep |q_gt| >= per-sample quantile threshold
# - qmag_min: keep |q_gt| > qmag_min
# (if both are set, they are combined with AND)

import torch
import seaborn as sns
import matplotlib.pyplot as plt

# Match experiment/downscaling_LMM_res_2mT_pretrain.yaml defaults
AT_QMAG_QUANTILE = None
AT_QMAG_MIN = 1.0e-15


def flux_vector_l1_mae_masked(
    T_pred: torch.Tensor,
    T_gt: torch.Tensor,
    qmag_quantile=None,
    qmag_min=None,
):
    """Masked componentwise MAE for anisotropic flux vectors (GT-mask semantics)."""
    qx_p, qy_p = field_losses._anisotropic_flux_q(T_pred, dx=AT_DX, dy=AT_DY)
    qx_g, qy_g = field_losses._anisotropic_flux_q(T_gt, dx=AT_DX, dy=AT_DY)

    # Per-pixel componentwise absolute error across flux vector components
    l1_map = 0.5 * (torch.abs(qx_p - qx_g) + torch.abs(qy_p - qy_g))

    mag_g = torch.hypot(qx_g, qy_g)
    mask = field_losses._build_anisotropic_qmag_mask(
        mag_g,
        qmag_quantile=qmag_quantile,
        qmag_min=qmag_min,
    )

    l1_mean = field_losses._masked_mean(l1_map, mask)
    mask_frac = float(mask.float().mean().item()) if mask is not None else 1.0
    return float(l1_mean.item()), mask_frac


ql1_metrics_list = []
ql1_mask_fracs = []
for model_name in metric_models:
    rows_2mt = results[model_name][results[model_name]['variable'] == '2mT']
    for _, row in rows_2mt.iterrows():
        ref_row = results['COSMO-CLM'][
            (results['COSMO-CLM']['time_step'] == row['time_step'])
            & (results['COSMO-CLM']['variable'] == '2mT')
        ]
        if ref_row.empty:
            continue

        ref_field = ref_row.iloc[0]['spat_distr']
        pred_field = row['spat_distr']
        ref_np = ref_field.numpy() if hasattr(ref_field, 'numpy') else ref_field
        pred_np = pred_field.numpy() if hasattr(pred_field, 'numpy') else pred_field

        T_gt = torch.as_tensor(ref_np, dtype=torch.float32).unsqueeze(0)
        T_pred = torch.as_tensor(pred_np, dtype=torch.float32).unsqueeze(0)

        with torch.no_grad():
            q_l1, mask_frac = flux_vector_l1_mae_masked(
                T_pred,
                T_gt,
                qmag_quantile=AT_QMAG_QUANTILE,
                qmag_min=AT_QMAG_MIN,
            )

        ql1_metrics_list.append({
            'model': model_name,
            'target_var': row['target_var'],
            'var': row['variable'],
            'metric': 'Q_vec_L1_MAE_masked',
            'value': q_l1,
        })
        ql1_mask_fracs.append(mask_frac)

if ql1_metrics_list:
    ql1_df = pd.DataFrame(ql1_metrics_list)
    metrics_df = pd.concat([metrics_df, ql1_df], ignore_index=True)
    metrics_df.to_pickle(output_path + './Our_inference_metrics_with_q_vec.pkl')

mean_mask_frac = np.mean(ql1_mask_fracs) if len(ql1_mask_fracs) > 0 else float('nan')
print(
    f'Added Q_vec_L1_MAE_masked ({len(ql1_metrics_list)} rows; lower = better). '
    f'Mean mask fraction kept: {mean_mask_frac:.4f} '
    f'(qmag_quantile={AT_QMAG_QUANTILE}, qmag_min={AT_QMAG_MIN}).'
)

ql1_plot_df = metrics_df[metrics_df['metric'] == 'Q_vec_L1_MAE_masked'].copy()
if len(ql1_plot_df) == 0:
    print('No Q_vec_L1_MAE_masked metrics found. Skipping plot.')
else:
    sns.set_theme(style='whitegrid', font_scale=1.3)
    g = sns.catplot(
        data=ql1_plot_df,
        kind='box',
        x='model',
        y='value',
        hue='model',
        height=5,
        aspect=1.5,
        legend=False,
        showmeans=True,
        meanprops={
            'marker': 'v',
            'markerfacecolor': 'white',
            'markeredgecolor': 'black',
            'markersize': 7,
        },
    )

    for ax in g.axes.flatten():
        ax.set_xlabel('')
        ax.set_ylabel('masked mean 0.5*(|delta qx| + |delta qy|)')
        ax.set_title('Flux vector MAE with GT |q| mask (2mT)')
        for tick in ax.get_xticklabels():
            tick.set_rotation(45)

    plt.tight_layout()
    plt.show()
